# DataMate: Qwen3.5-9B Q8_0 trên Kaggle (2 x T4)

Trước khi chạy: **Settings → Accelerator: GPU T4 x2**, **Internet: On** (tài khoản cần xác minh số điện thoại).
Chạy lần lượt từng ô. Ô cuối in ra **2 đường link** (mỗi GPU một server): copy cả 2 gửi lại. Giữ tab mở; ô cuối chạy mãi để phiên không bị tắt.

Model: Qwen3.5-9B bản GGUF Q8_0 (8-bit, gần như y hệt bản đầy đủ), chạy bằng ollama với template tắt chế độ suy nghĩ,
giống hệt server trên gpu156 (chỉ khác Q4 -> Q8). Mỗi GPU một server, mỗi server 2 lượt song song.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
%%bash
set -e
# script cài ollama mới cần zstd để giải nén
if ! command -v zstd > /dev/null; then
  (apt-get update -qq && apt-get install -y -qq zstd) > /tmp/zstd_install.log 2>&1 || (tail -n 20 /tmp/zstd_install.log; exit 1)
fi
zstd --version
curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama_install.log 2>&1 || (tail -n 20 /tmp/ollama_install.log; exit 1)
ollama --version
wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
chmod +x /usr/local/bin/cloudflared
cloudflared --version

In [ ]:
import os, subprocess, time, json, re, urllib.request
D = "/tmp/datamate"
os.makedirs(D, exist_ok=True)
F = f"{D}/Qwen3.5-9B-Q8_0.gguf"
SIZE = 9527502048
if not os.path.exists(F) or os.path.getsize(F) != SIZE:
    subprocess.run(["curl", "-L", "-f", "-C", "-", "-o", F,
                    "https://huggingface.co/unsloth/Qwen3.5-9B-GGUF/resolve/main/Qwen3.5-9B-Q8_0.gguf"], check=True)
print("model:", os.path.getsize(F), "byte", "OK" if os.path.getsize(F) == SIZE else "THIẾU")

In [ ]:
MODELFILE = '''FROM __F__
TEMPLATE """{{- range .Messages }}<|im_start|>{{ .Role }}
{{ .Content }}<|im_end|>
{{ end }}<|im_start|>assistant
<think>

</think>

"""
PARAMETER num_ctx 16384
PARAMETER num_predict 1000
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"
PARAMETER repeat_penalty 1
PARAMETER top_p 1
PARAMETER top_k 0
'''.replace("__F__", F)
open(f"{D}/Modelfile", "w").write(MODELFILE)

PORTS = (11434, 11435)
procs = []
for g, port in enumerate(PORTS):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), OLLAMA_HOST=f"127.0.0.1:{port}", OLLAMA_MODELS=f"{D}/models",
               OLLAMA_NUM_PARALLEL="2", OLLAMA_KEEP_ALIVE="-1", OLLAMA_CONTEXT_LENGTH="16384")
    procs.append(subprocess.Popen(["ollama", "serve"], env=env, stdout=open(f"{D}/ollama_{g}.log", "w"),
                                  stderr=subprocess.STDOUT, start_new_session=True))
time.sleep(10)
for port in PORTS:
    subprocess.run(["ollama", "create", "qwen35q8", "-f", f"{D}/Modelfile"], env=dict(os.environ, OLLAMA_HOST=f"127.0.0.1:{port}"),
                   check=True, stdout=subprocess.DEVNULL)
print("đã tạo model qwen35q8 trên", PORTS)

In [ ]:
# thử mỗi server một câu (lần đầu nạp model lên GPU, mất 1-2 phút)
for port in PORTS:
    body = {"model": "qwen35q8", "temperature": 0, "max_tokens": 30, "seed": 2026,
            "messages": [{"role": "system", "content": "Trả lời đúng một con số."}, {"role": "user", "content": "1+1 bằng mấy?"}]}
    req = urllib.request.Request(f"http://127.0.0.1:{port}/v1/chat/completions", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    t = time.time()
    out = json.loads(urllib.request.urlopen(req, timeout=900).read())
    print(port, f"{time.time() - t:.1f}s:", repr(out["choices"][0]["message"]["content"]))
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv

In [ ]:
urls = []
for port in PORTS:
    log = f"{D}/tunnel_{port}.log"
    subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{port}", "--http-host-header", "localhost"],
                     stdout=open(log, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        time.sleep(2)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log).read())
        if m:
            urls.append(m.group(0))
            break
print("GỬI 2 LINK NÀY:")
print(",".join(urls))

In [ ]:
# giữ phiên chạy; mỗi 5 phút in trạng thái (server còn sống thì poll() là None)
while True:
    time.sleep(300)
    print(time.strftime("%H:%M"), [p.poll() for p in procs], flush=True)